# Maternal mood (Month 3) vs smile synchrony (Month 4)


In [ ]:
from notebook_setup import init
FINAL_DIR, PDF_DIR = init()
import plot_style
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

from analysis_config import (
    ALPHA,
    BASE_PATH,
    OPENFACE_PATH_ADD,
    QUESTIONNAIRE_DIR,
    SMILE_AUS,
    QUESTIONNAIRE_SCORE_COLS,
    QUESTIONNAIRE_SCORE_LABELS,
)
from dyad_analysis_utils import prepare_au_dataframe, merge_openface_paths, pair_corr

SCORE_COLS = QUESTIONNAIRE_SCORE_COLS
SCORE_LABELS = QUESTIONNAIRE_SCORE_LABELS

print(f"OpenFace: {BASE_PATH}")
if OPENFACE_PATH_ADD.exists():
    print(f"Extra OpenFace: {OPENFACE_PATH_ADD}")

In [ ]:
epds = pd.read_csv(QUESTIONNAIRE_DIR / "...", sep=";")
stai = pd.read_csv(QUESTIONNAIRE_DIR / "...", sep=";")

questionnaires = (
    epds[["diti_nr", "total"]].rename(columns={"total": "EPDS_total"})
    .merge(
        stai[["diti_nr", "X_1", "X_2"]].rename(columns={"X_1": "STAI_X1", "X_2": "STAI_X2"}),
        on="diti_nr",
        how="inner",
    )
)
print(f"Questionnaire records: {len(questionnaires)}")

In [ ]:
meeting_groups = merge_openface_paths(BASE_PATH, OPENFACE_PATH_ADD)

rows = []
for meeting_id, sessions in meeting_groups.items():
    if not sessions:
        continue
    try:
        meeting_nr = int(meeting_id.split("_")[1])
    except (IndexError, ValueError):
        continue

    infant_path, adult_path = sessions[0]
    raw_infant = pd.read_csv(infant_path)
    raw_adult = pd.read_csv(adult_path)
    raw_infant.columns = raw_infant.columns.str.strip()
    raw_adult.columns = raw_adult.columns.str.strip()

    if any(au not in raw_infant.columns or au not in raw_adult.columns for au in SMILE_AUS):
        continue

    min_len = min(len(raw_infant), len(raw_adult))
    raw_infant = raw_infant.iloc[:min_len]
    raw_adult = raw_adult.iloc[:min_len]

    infant_df = prepare_au_dataframe(raw_infant, SMILE_AUS)
    adult_df = prepare_au_dataframe(raw_adult, SMILE_AUS)

    valid = infant_df[SMILE_AUS].notna().all(axis=1) & adult_df[SMILE_AUS].notna().all(axis=1)
    inf_v = infant_df.loc[valid, SMILE_AUS]
    adu_v = adult_df.loc[valid, SMILE_AUS]

    rows.append({
        "diti_nr": meeting_nr,
        "n_frames": len(inf_v),
        "sync_composite": pair_corr(inf_v.mean(axis=1), adu_v.mean(axis=1)),
    })

df = pd.DataFrame(rows)
df = df.merge(questionnaires, on="diti_nr", how="inner")
print(f"Month-4 meetings with synchrony and questionnaire scores: {len(df)}")

In [ ]:
LINE_COLOR = "#D55E00"
METRIC_COL = "sync_composite"

plot_stats = []
for scol, panel_label in zip(SCORE_COLS, SCORE_LABELS):
    sub = df[[METRIC_COL, scol]].dropna()
    r, p = pearsonr(sub[METRIC_COL], sub[scol]) if len(sub) >= 3 else (np.nan, np.nan)
    plot_stats.append({"score_col": scol, "label": panel_label, "r": r, "p": p, "n": len(sub)})

plot_df = pd.DataFrame(plot_stats)
reject, p_holm, _, _ = multipletests(plot_df["p"].values, alpha=ALPHA, method="holm")
plot_df["p_holm"] = p_holm
plot_df["significant_holm"] = reject

print("Composite synchrony vs questionnaires (Holm correction across 3 tests):")
display(plot_df[["label", "r", "p", "p_holm", "significant_holm", "n"]].round(4))

fig, axes = plt.subplots(1, len(SCORE_COLS), figsize=(10, 3.5), squeeze=False)
plot_style.fig_suptitle(
    fig,
    "Composite synchrony (AU06+AU12)/2 vs questionnaires",
    fontweight="bold",
    y=1.12,
)

for j, row in plot_df.iterrows():
    ax = axes[0, j]
    scol = row["score_col"]
    sub = df[[METRIC_COL, scol]].dropna()
    x = sub[METRIC_COL].values
    y = sub[scol].values
    ax.scatter(x, y, alpha=0.75, edgecolors="white", s=45)
    if len(x) >= 3:
        z = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, np.poly1d(z)(xs), linestyle="--", color=LINE_COLOR, alpha=0.9)
        stars = plot_style.pvalue_stars(row["p_holm"])
        p_text = plot_style.format_p_value(row["p_holm"])
        ax.text(
            0.97,
            0.97,
            f"r = {row['r']:.2f}\np = {p_text} (Holm){stars}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=12,
        )
    ax.set_title(row["label"], fontsize=12, fontweight="bold", pad=8)
    ax.set_xlabel("Synchrony")
    if j == 0:
        ax.set_ylabel("Questionnaire score")

plt.tight_layout()
plot_style.save_pdf(fig, "questionnaires_composite_sync", PDF_DIR)
plot_style.finish_figure(fig)


In [ ]:
EPDS_CUTOFF = 13
TOP_N = 25

ranked = (
    df.dropna(subset=[METRIC_COL, "EPDS_total"])
    .sort_values(METRIC_COL, ascending=False)
    .reset_index(drop=True)
)
ranked["sync_rank"] = ranked.index + 1
above = ranked[ranked["EPDS_total"] > EPDS_CUTOFF]
print(f"Mothers with EPDS > {EPDS_CUTOFF}: {len(above)}")
print(f"  their synchrony ranks (1 = highest, out of {len(ranked)}): {above['sync_rank'].tolist()}")
print(f"Max EPDS among the {TOP_N} dyads with the highest synchrony: {ranked.head(TOP_N)['EPDS_total'].max()}")
